In [1]:
import numpy as np
import scipy as sp
import xarray as xr
import seaborn as sea
import matplotlib.pyplot as plt

# Codebook 

**Contains:**

- Histograms of variables --> mean of a year of data
- Histograms of variables --> seasonal variations

In [2]:
norkyst_gen_path = '/lustre/storeB/project/fou/hi/foccus/datasets/symlinks/norkystv3-hindcast/2024'
#choose all
import glob
filenames = glob.glob(f'{norkyst_gen_path}/*.nc')
ds = xr.open_mfdataset(filenames, engine = 'netcdf4').isel(s_rho = -1, s_w = -1)

In [3]:
#mem error - need to do it per variable maybe
salinity = ds.salinity 
temperature = ds.temperature 
v_northward = ds.v_northward
u_eastward = ds.u_eastward
zeta = ds.zeta
ds_monthly_s = salinity.resample(time = 'MS').mean().compute()
ds_monthly_t = temperature.resample(time = 'MS').mean().compute()
ds_monthly_v = v_northward.resample(time = 'MS').mean().compute()
ds_monthly_u = u_eastward.resample(time = 'MS').mean().compute()
ds_monthly_z = zeta.resample(time = 'MS').mean().compute() 

In [4]:
yearly_avg_s = ds_monthly_s.resample(time = 'Y').mean().compute()
yearly_avg_t = ds_monthly_t.resample(time = 'Y').mean().compute()
yearly_avg_v = ds_monthly_v.resample(time = 'Y').mean().compute()
yearly_avg_u = ds_monthly_u.resample(time = 'Y').mean().compute()
yearly_avg_z = ds_monthly_z.resample(time = 'Y').mean().compute()

/lustre/storeB/project/fou/hi/foccus/.venv/lib64/python3.11/site-packages/xarray/groupers.py:487: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  self.index_grouper = pd.Grouper(
/lustre/storeB/project/fou/hi/foccus/.venv/lib64/python3.11/site-packages/xarray/groupers.py:487: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  self.index_grouper = pd.Grouper(
/lustre/storeB/project/fou/hi/foccus/.venv/lib64/python3.11/site-packages/xarray/groupers.py:487: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  self.index_grouper = pd.Grouper(
/lustre/storeB/project/fou/hi/foccus/.venv/lib64/python3.11/site-packages/xarray/groupers.py:487: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  self.index_grouper = pd.Grouper(
/lustre/storeB/project/fou/hi/foccus/.venv/lib64/python3.11/site-package

In [18]:
#to use in function - we collect them into one dataset without having to change the function
ds_final_yearly = xr.merge([yearly_avg_s, yearly_avg_t, yearly_avg_v, yearly_avg_u, yearly_avg_z])#, dim = 'time')
print(ds_final_yearly)
#ds_final_yearly.to_zarr('/lustre/storeB/project/fou/hi/foccus/malene/ocean-ai/plot/master/statistics/saved_nc/yearly_average.zarr', mode="w")
#Trying to save the yearly concatted file for later use - because of long run time it is ideal to just save it straight ahead
ds_final_yearly.to_netcdf(f'/lustre/storeB/project/fou/hi/foccus/malene/ocean-ai/plot/master/statistics/saved_nc/yearly_average.nc')
#Also, it would be ideal to save the monthly files as well so we repeat - in case no mem-error we will then use this to create the seasonal variation plots 
ds_final_monthly = xr.merge([ds_monthly_s, ds_monthly_t, ds_monthly_v, ds_monthly_u, ds_monthly_z])#, dim = 'time')
ds_final_monthly.to_netcdf(f'/lustre/storeB/project/fou/hi/foccus/malene/ocean-ai/plot/master/statistics/saved_nc/monthly_average.nc')
#print(ds_final_yearly.nbytes / 1e6, 'MB')
#print(type(ds_final_yearly[list(ds_final_yearly.data_vars)[0]].data))
print(ds_final_monthly.nbytes / 1e6, 'MB')
#print(type(ds_final_monthly[list(ds_final_monthly.data_vars)[0]].data))

<xarray.Dataset> Size: 114MB
Dimensions:      (X: 2747, Y: 1148, time: 1)
Coordinates:
    s_rho        float64 8B -0.004904
  * X            (X) float64 22kB 0.0 800.0 1.6e+03 ... 2.196e+06 2.197e+06
  * Y            (Y) float64 9kB 0.0 800.0 1.6e+03 ... 9.168e+05 9.176e+05
    s_w          float64 8B 0.0
    lon          (Y, X) float64 25MB 8.7 8.706 8.711 8.717 ... 18.29 18.31 18.33
    lat          (Y, X) float64 25MB 54.29 54.3 54.31 ... 75.71 75.72 75.73
  * time         (time) datetime64[ns] 8B 2024-12-31
Data variables:
    salinity     (time, Y, X) float32 13MB nan nan nan nan ... 34.37 34.36 34.36
    temperature  (time, Y, X) float32 13MB nan nan nan nan ... 2.282 2.285 2.278
    v_northward  (time, Y, X) float32 13MB 0.0 0.0 0.0 ... -0.008931 -0.008931
    u_eastward   (time, Y, X) float32 13MB 0.0 0.0 0.0 ... -0.04572 -0.04572
    zeta         (time, Y, X) float32 13MB nan nan nan ... -0.5105 -0.5105
Attributes:
    grid:                   grid
    location:               

In [ ]:
#check if code from summer project is adaptable for this use case
sea.set_theme(style="darkgrid")
def histograms_with_specified_layout(ds, type_of_avg, time_to_check = 1, save_path = 'figures/histogram.png'):
    """
    Definition: 

    Arguments:
    arg[1] : dataset - the inference dataset 
    arg[2] : time to check (int) - the timestep in the data would you like the check the variable distribution for

    Outputs: 
    Figure of the variable distributions 
    """

    variables = ['salinity', 'u_eastward', 'v_northward', 'temperature', 'zeta']
    data = {var : ds[var].isel(time = time_to_check).values.ravel() for var in variables}
    data = {var : values[~np.isnan(values)] for var, values in data.items()}

    fig = plt.figure(figsize = (12,7))
    titles = ['Salinity', 'U-Eastward', 'V-Northward', 'temperature', 'zeta']
    """ 
    Trying to make the plot more visually appealing witg gridspec, to elongate the bottom plots.
    """
    gs = fig.add_gridspec(2,6)
    axes = [fig.add_subplot(gs[0,0:2]), fig.add_subplot(gs[0,2:4]), fig.add_subplot(gs[0,4:6]), fig.add_subplot(gs[1,0:3]), fig.add_subplot(gs[1,3:6])]
    
    for i, (var, values) in enumerate(data.items()):
        row, col = divmod(i,3)
        sea.histplot(values, bins = 30, kde = True, ax = axes[i], stat='density', color = 'royalblue', edgecolor = 'darkolivegreen')
        axes[i].set_title(titles[i], fontsize = 15)
        axes[i].set_facecolor("#C4C6D5")
        #axes[1,2].axis('off')

    plt.suptitle(f'Histogram & Density curve for Norkyst variables - {type_of_avg} average')
    plt.tight_layout()
    fig.savefig(save_path)

In [ ]:
histograms_with_specified_layout(ds_final_yearly, f'Yearly', time_to_check=0, save_path=f'figures/yearly_avg.png')

In [ ]:
#Now for the seasonal histograms 
"""
def seasons(ds):
    #First - create monthly means, by resampling the data to include monthly means gathered at the start of each month
    daily_avg = ds.resample(time = 'D').mean()
    monthly_avg = daily_avg.resample(time = 'MS').mean()
    #Then we make weights based on the number of days in the respective months
    month_length = monthly_avg.time.dt.days_in_month
    #Preview
    print(month_length)
    #Weights - tid per måned / summen av alle tider  
    weights = month_length.groupby('time.season') / month_length.groupby('time.season').sum()
    #testing 
    if np.testing.assert_allclose(weights.groupby('time.season').sum().values, np.ones(4)) != 1.0:
        raise ValueError(f'Weights are not properly defined - please ensure that the sum of weights for the seasons equal 1.0')
    #apply weights to dataset
    ds_weighted = (ds*weights).groupby('time.season').sum(dim = 'time')
    return ds_weighted 
"""